In [6]:
import ipywidgets as widgets
from IPython.display import display, Math, HTML

style = {'description_width': '140px'}
l_layout = widgets.Layout(width='250px')

b_in = widgets.IntText(value=275, description='Laius b (mm):', style=style, layout=l_layout)
h_in = widgets.IntText(value=450, description='Kõrgus h (mm):', style=style, layout=l_layout)
Es_in = widgets.FloatText(value=200000, description='E_s (N/mm²):', style=style, layout=l_layout)
Ecm_in = widgets.FloatText(value=31000, description='E_cm (N/mm²):', style=style, layout=l_layout)
fctm_in = widgets.FloatText(value=2.60, description='f_ctm (N/mm²):', style=style, layout=l_layout)

As1_in = widgets.IntText(value=402, description='A_s1 tõmbe (mm²):', style=style, layout=l_layout)
As2_in = widgets.IntText(value=0, description='A_s2 surve (mm²):', style=style, layout=l_layout)
ds1_in = widgets.IntText(value=415, description='d_s1 (mm):', style=style, layout=l_layout)
ds2_in = widgets.IntText(value=0, description='d_s2 (mm):', style=style, layout=l_layout)

N_in = widgets.FloatText(value=0.00, description='Pikijõud N (kN):', style=style, layout=l_layout)
M_in = widgets.FloatText(value=51.30, description='Moment M (kNm):', style=style, layout=l_layout)
fi_in = widgets.FloatText(value=2.2, description='Roometegur φ:', style=style, layout=l_layout)

vasak_tulp = widgets.VBox([
    widgets.HTML("<h3 style='margin-top:0; color:#2c3e50;'>Eurokoodi sisendid</h3>"),
    widgets.HTML("<h4>Geomeetria</h4>"), b_in, h_in, Es_in, Ecm_in, fctm_in,
    widgets.HTML("<h4>Sarrustus</h4>"), As1_in, As2_in, ds1_in, ds2_in,
    widgets.HTML("<h4>Koormused</h4>"), N_in, M_in, fi_in
], layout=widgets.Layout(padding='15px', border='1px solid #ccc', background_color='#f9f9f9', min_width='280px'))

print("Osa 1: Sisendid laetud!")


Osa 1: Sisendid laetud!


In [7]:
def täielik_eurokoodi_arvutus(b, h, Es, Ecm, fctm, As1, As2, ds1, ds2, N_v, M_v, fi):
    N = N_v * 1000       
    M = M_v * 1000000    
    Ac = b * h
    Soc = b * h * (h / 2)
    Ic = (b * h**3) / 12
    
    # --- STADIUM I: ENNE ROOMET ---
    alpha_s1 = Es / Ecm
    Ared1 = Ac + alpha_s1 * As1 + alpha_s1 * As2
    S0red1 = Soc + alpha_s1 * As1 * (h - ds1) + alpha_s1 * As2 * (h - ds2)
    y0red1 = S0red1 / Ared1 if Ared1 != 0 else h/2
    x1 = h - y0red1
    ys1_1 = ds1 - x1
    ys2_1 = y0red1 - ds2
    Ired1 = Ic + Ac * (y0red1 - h/2)**2 + alpha_s1 * As1 * (ds1 - x1)**2 + alpha_s1 * As2 * (x1 - ds2)**2
    Wcred1 = Ired1 / x1 if x1 != 0 else 1e-9
    Wtred1 = Ired1 / y0red1 if y0red1 != 0 else 1e-9
    sigma_c_max1 = -N/Ared1 - M/Wcred1
    sigma_ct_max1 = -N/Ared1 + M/Wtred1
    sigma_s1_1 = alpha_s1 * (-N/Ared1 + M * (ds1 - x1) / Ired1)
    sigma_s2_1 = alpha_s1 * (-N/Ared1 - M * (x1 - ds2) / Ired1)
    Mcr1 = fctm * Wtred1 / 1000000
    
    # --- STADIUM II: PÄRAST ROOMET ---
    Ec_ef = (1.05 * Ecm) / (1 + fi)
    alpha_s2 = Es / Ec_ef
    Ared2 = Ac + alpha_s2 * As1 + alpha_s2 * As2
    S0red2 = Soc + alpha_s2 * As1 * (h - ds1) + alpha_s2 * As2 * (h - ds2)
    y0red2 = S0red2 / Ared2 if Ared2 != 0 else h/2
    x2 = h - y0red2
    ys1_2 = ds1 - x2
    ys2_2 = y0red2 - ds2
    Ired2 = Ic + Ac * (y0red2 - h/2)**2 + alpha_s2 * As1 * (ds1 - x2)**2 + alpha_s2 * As2 * (x2 - ds2)**2
    Wcred2 = Ired2 / x2 if x2 != 0 else 1e-9
    Wtred2 = Ired2 / y0red2 if y0red2 != 0 else 1e-9
    sigma_c_max2 = -N/Ared2 - M/Wcred2
    sigma_ct_max2 = -N/Ared2 + M/Wtred2
    sigma_s1_2 = alpha_s2 * (-N/Ared2 + M * (ds1 - x2) / Ired2)
    sigma_s2_2 = alpha_s2 * (-N/Ared2 - M * (x2 - ds2) / Ired2)
    Mcr2 = fctm * Wtred2 / 1000000

    # --- STADIUM III: PRAGUNENUD ENNE ROOMET ---
    A_eff3 = alpha_s1 * (As1 + As2)
    B_eff3 = 2 * alpha_s1 * (As1 * ds1 + As2 * ds2)
    x_cr3 = (-A_eff3 + (A_eff3**2 + b * B_eff3)**0.5) / b if b != 0 else 0
    Icr3 = (b * x_cr3**3)/3 + alpha_s1 * As1 * (ds1 - x_cr3)**2 + alpha_s1 * As2 * (x_cr3 - ds2)**2
    sigma_c_max3 = -M * x_cr3 / Icr3 if Icr3 != 0 else 0
    sigma_s1_3 = alpha_s1 * (M * (ds1 - x_cr3) / Icr3) if Icr3 != 0 else 0
    sigma_s2_3 = alpha_s1 * (M * (ds2 - x_cr3) / Icr3) if Icr3 != 0 else 0

    # --- STADIUM IV: PRAGUNENUD PÄRAST ROOMET ---
    A_eff4 = alpha_s2 * (As1 + As2)
    B_eff4 = 2 * alpha_s2 * (As1 * ds1 + As2 * ds2)
    x_cr4 = (-A_eff4 + (A_eff4**2 + b * B_eff4)**0.5) / b if b != 0 else 0
    Icr4 = (b * x_cr4**3)/3 + alpha_s2 * As1 * (ds1 - x_cr4)**2 + alpha_s2 * As2 * (x_cr4 - ds2)**2
    sigma_c_max4 = -M * x_cr4 / Icr4 if Icr4 != 0 else 0
    sigma_s1_4 = alpha_s2 * (M * (ds1 - x_cr4) / Icr4) if Icr4 != 0 else 0
    sigma_s2_4 = alpha_s2 * (M * (ds2 - x_cr4) / Icr4) if Icr4 != 0 else 0

    kuva_pikk_raport(b, h, Ac, Soc, Ic, alpha_s1, Ared1, S0red1, y0red1, x1, Ired1, Wcred1, Wtred1, fi, Ec_ef, alpha_s2, Ared2, S0red2, y0red2, x2, Ired2, Wcred2, Wtred2, x_cr3, Icr3, x_cr4, Icr4, sigma_c_max1, sigma_c_max2, sigma_c_max3, sigma_c_max4, sigma_ct_max1, sigma_ct_max2, sigma_s1_1, sigma_s1_2, sigma_s1_3, sigma_s1_4, sigma_s2_1, sigma_s2_2, sigma_s2_3, sigma_s2_4, Mcr1, Mcr2, Es, Ecm, fctm, As1, As2, ds1, ds2, N, M)

print("Osa 2: Arvutused salvestatud!")


Osa 2: Arvutused salvestatud!


In [8]:
def kuva_pikk_raport(b, h, Ac, Soc, Ic, alpha_s1, Ared1, S0red1, y0red1, x1, Ired1, Wcred1, Wtred1, fi, Ec_ef, alpha_s2, Ared2, S0red2, y0red2, x2, Ired2, Wcred2, Wtred2, x_cr3, Icr3, x_cr4, Icr4, sigma_c_max1, sigma_c_max2, sigma_c_max3, sigma_c_max4, sigma_ct_max1, sigma_ct_max2, sigma_s1_1, sigma_s1_2, sigma_s1_3, sigma_s1_4, sigma_s2_1, sigma_s2_2, sigma_s2_3, sigma_s2_4, Mcr1, Mcr2, Es, Ecm, fctm, As1, As2, ds1, ds2, N, M):
    print("=========================================================================================")
    print("                      EUROKOODI KOHANE DETAISED VALEMID JA ASENDUSED                     ")
    print("=========================================================================================\n")
    
    print("### 1. PRAGUNEMATA RISTLÕIGE ENNE ROOMET")
    display(Math(rf'\alpha_s = \frac{{E_s}}{{E_{{cm}}}} = \frac{{{Es:.0f}}}{{{Ecm:.0f}}} = {alpha_s1:.2f}'))
    display(Math(rf'A_{{red}} = {b} \times {h} + {alpha_s1:.2f} \times {As1} + {alpha_s1:.2f} \times {As2} = {Ared1:.0f} \text{{ mm}}^2'))
    display(Math(rf'S_{{0,red}} = \left({b} \times {h} \times \frac{{{h}}}{{2}}\right) + {alpha_s1:.2f} \times {As1} \times ({h} - {ds1}) = {S0red1/10**6:.2f} \times 10^6 \text{{ mm}}^3'))
    display(Math(rf'y_{{0,red}} = \frac{{{S0red1/10**6:.2f} \times 10^6}}{{{Ared1:.0f}}} = {y0red1:.0f} \text{{ mm}} \implies x = {h} - {y0red1:.0f} = {x1:.0f} \text{{ mm}}'))
    display(Math(rf'I_{{red}} = \frac{{{b} \times {h}^3}}{{12}} + A_c \left({y0red1:.0f} - \frac{{{h}}}{{2}}\right)^2 + \alpha_s A_{{s1}} y_{{s1}}^2 = {Ired1/10**6:.2f} \times 10^6 \text{{ mm}}^4'))
    display(Math(rf'W_{{c,red}} = \frac{{{Ired1/10**6:.2f} \times 10^6}}{{{x1:.0f}}} = {Wcred1/10**6:.2f} \times 10^6 \text{{ mm}}^3 \quad W_{{t,red}} = {Wtred1/10**6:.2f} \times 10^6 \text{{ mm}}^3'))
    display(Math(rf'\sigma_{{c,max}} = -\frac{{{N:.0f}}}{{{Ared1:.0f}}} - \frac{{{M/10**6:.2f} \times 10^6}}{{{Wcred1/10**6:.2f} \times 10^6}} = {sigma_c_max1:.2f} \text{{ N/mm}}^2'))
    display(Math(rf'\sigma_{{ct,max}} = -\frac{{{N:.0f}}}{{{Ared1:.0f}}} + \frac{{{M/10**6:.2f} \times 10^6}}{{{Wtred1/10**6:.2f} \times 10^6}} = {sigma_ct_max1:.2f} \text{{ N/mm}}^2'))
    display(Math(rf'M_{{cr}} = {fctm} \times {Wtred1/10**6:.2f} \times 10^6 = {Mcr1:.2f} \text{{ kNm}}'))

    print("\n### 2. PRAGUNEMATA RISTLÕIGE PÄRAST ROOMET")
    display(Math(rf'E_{{c,ef}} = \frac{{1,05 \times {Ecm:.0f}}}{{1 + {fi}}} = {Ec_ef:.0f} \text{{ N/mm}}^2 \implies \alpha_s = \frac{{{Es:.0f}}}{{{Ec_ef:.0f}}} = {alpha_s2:.2f}'))
    display(Math(rf'A_{{red,t}} = {Ared2:.0f} \text{{ mm}}^2 \quad y_{{0,red,t}} = {y0red2:.0f} \text{{ mm}} \quad x_t = {x2:.0f} \text{{ mm}}'))
    display(Math(rf'I_{{red,t}} = {Ired2/10**6:.2f} \times 10^6 \text{{ mm}}^4 \implies M_{{cr}} = {Mcr2:.2f} \text{{ kNm}}'))

    print("\n### 3. PRAGUNENUD RISTLÕIGE ENNE ROOMET")
    display(Math(rf'x = -\frac{{{alpha_s1:.2f} \times ({As1} + {As2})}}{{{b}}} + \sqrt{{\left(\frac{{{alpha_s1:.2f} \times {As1}}}{{{b}}}\right)^2 + \frac{{2 \times {alpha_s1:.2f} \times ({As1} \times {ds1})}}{{{b}}}}} = {x_cr3:.1f} \text{{ mm}}'))
    display(Math(rf'I_{{cr,red}} = \frac{{{b} \times {x_cr3:.1f}^3}}{{3}} + {alpha_s1:.2f} \times {As1} \times ({ds1} - {x_cr3:.1f})^2 = {Icr3/10**6:.2f} \times 10^6 \text{{ mm}}^4'))
    display(Math(rf'\sigma_{{c,max}} = -\frac{{{M/10**6:.2f} \times 10^6 \times {x_cr3:.1f}}}{{{Icr3/10**6:.2f} \times 10^6}} = {sigma_c_max3:.2f} \text{{ N/mm}}^2'))
    display(Math(rf'\sigma_{{s1}} = {alpha_s1:.2f} \times \frac{{{M/10**6:.2f} \times 10^6 \times ({ds1} - {x_cr3:.1f})}}{{{Icr3/10**6:.2f} \times 10^6}} = {sigma_s1_3:.2f} \text{{ N/mm}}^2'))

    print("\n### 4. PRAGUNENUD RISTLÕIGE PÄRAST ROOMET")
    display(Math(rf'x_t = {x_cr4:.1f} \text{{ mm}} \implies I_{{cr,red,t}} = {Icr4/10**6:.2f} \times 10^6 \text{{ mm}}^4'))
    display(Math(rf'\sigma_{{c,max,t}} = {sigma_c_max4:.2f} \text{{ N/mm}}^2 \quad \sigma_{{s1,t}} = {sigma_s1_4:.2f} \text{{ N/mm}}^2'))

    # KOONDTABEL HÄSTI LOETAVAS DISAINIS
    html_table = f"""
    <br><br><h3 style='color:#1e6823;'>KOKKUVÕTE: PINGETE JA PARAMEETRITE KOONDTABEL [MPa]</h3>
    <table border="1" cellpadding="5" style="border-collapse:collapse; text-align:center; width:100%; border:2px solid #333;">
        <tr style="background-color:#2e7d32; color:white; font-weight:bold;">
            <th>Mõõdetud pinge / parameeter</th>
            <th style="background-color:#1b5e20;">Pragunemata Enne roomet</th>
            <th style="background-color:#1b5e20;">Pragunemata Pärast roomet</th>
            <th style="background-color:#b71c1c;">Pragunenud Enne roomet</th>
            <th style="background-color:#b71c1c;">Pragunenud Pärast roomet</th>
        </tr>
        <tr>
            <td><b>&sigma;<sub>c,max</sub></b> (Betooni surve)</td>
            <td>{sigma_c_max1:.2f}</td><td>{sigma_c_max2:.2f}</td>
            <td style="color:red; font-weight:bold;">{sigma_c_max3:.2f}</td><td style="color:red; font-weight:bold;">{sigma_c_max4:.2f}</td>
        </tr>
        <tr>
            <td><b>&sigma;<sub>ct,max</sub></b> (Betooni tõmme)</td>
            <td>{sigma_ct_max1:.2f}</td><td>{sigma_ct_max2:.2f}</td><td>-</td><td>-</td>
        </tr>
        <tr>
            <td><b>&sigma;<sub>s1</sub></b> (Tõmbearmatuur)</td>
            <td>{sigma_s1_1:.2f}</td><td>{sigma_s1_2:.2f}</td><td>{sigma_s1_3:.2f}</td><td>{sigma_s1_4:.2f}</td>
        </tr>
        <tr>
            <td><b>&sigma;<sub>s2</sub></b> (Survearmatuur)</td>
            <td>{sigma_s2_1:.2f}</td><td>{sigma_s2_2:.2f}</td><td>{sigma_s2_3:.2f}</td><td>{sigma_s2_4:.2f}</td>
        </tr>
        <tr style="background-color:#fff9c4; font-weight:bold;">
            <td>M<sub>cr</sub> (Pragunemismoment, kNm)</td>
            <td>{Mcr1:.2f}</td><td>{Mcr2:.2f}</td><td>-</td><td>-</td>
        </tr>
    </table>
    """
    display(HTML(html_table))


In [9]:
keskosa = widgets.interactive_output(täielik_eurokoodi_arvutus, {
    'b': b_in, 'h': h_in, 'Es': Es_in, 'Ecm': Ecm_in, 'fctm': fctm_in,
    'As1': As1_in, 'As2': As2_in, 'ds1': ds1_in, 'ds2': ds2_in,
    'N_v': N_in, 'M_v': M_in, 'fi': fi_in
})

kogu_leht = widgets.HBox([vasak_tulp, keskosa], layout=widgets.Layout(flex_flow='row wrap', width='100%'))
display(kogu_leht)
